# Fig. 11: cross-city generalization

Run cells in order from a fresh kernel. Settings are in `configs/experiments.json`; artifacts use the paths described in the README. Timings depend on hardware and software.


In [ ]:
from fthbf.runtime import (DATA_DIR, CHECKPOINT_DIR, checkpoint_path, load_artifact,
                           result_dir, read_config, seed_everything, record_run)
import torch
cfg = read_config("experiments.json")["Fig06"]
seed_everything(cfg["seed"])
record_run("Fig06", cfg)


In [ ]:
with torch.no_grad():
    # =====================================================
    # Mean Optimized WSR across 4 scenario datasets
    # 6 algorithms × 4 scenarios → grouped bar chart
    # =====================================================

    import os
    import time
    import torch
    import numpy as np
    import matplotlib.pyplot as plt
    from tqdm import tqdm

    from solutions.trihybrid_WMMSE import TriHybridWMMSE, GNN
    from solutions.utils import computeWSR, channelDataset, HybridPrecodingBFGS, checkHybridConstraints
    from benchmark_algorithms.AOMO_alg import AOMOBeamformer
    from benchmark_algorithms.Decomp_alg import DecompBeamformer


    # =====================================================
    # Global settings 
    # =====================================================
    cdtype = torch.cfloat
    Nrf = cfg["rf_chains"]
    UWMMSE_I_LIST = cfg["unfolded_layers"]

    RESULT_DIR = result_dir("results_generalization_barplot")
    os.makedirs(RESULT_DIR, exist_ok=True)

    SEED = cfg["seed"]
    torch.manual_seed(SEED)
    np.random.seed(SEED)


    # =====================================================
    # Scenario datasets
    # =====================================================
    # DATA_DIR comes from the shared runtime configuration.

    SCENARIOS = {
        "Miami": "probInsts_Miami.pt",
        "Austin": "probInsts_Austin.pt",
        "Columbus": "probInsts_Columbus.pt",
        "Dallas": "probInsts_Dallas.pt",
    }

    PARAM_FILE = "parameters.pt"


    # =====================================================
    # Load global parameters
    # =====================================================
    parameters = load_artifact(os.path.join(DATA_DIR, PARAM_FILE))

    Nt = parameters["Nt"]
    Nc = parameters["Nc"]
    P = parameters["P"]
    Mk = parameters["Mk"]
    eta = parameters["eta"]
    Ns_k = parameters["Ns_k"]
    sigma2 = parameters["sigma2"]


    # =====================================================
    # Load UWMMSE models
    # =====================================================
    def load_uwmmse_models(I_max, example_H):
        in_dim = 2 * example_H.numel() + 1

        GNNs_gamma = torch.nn.ModuleList()
        GNNs_u = torch.nn.ModuleList()

        for _ in range(I_max):
            GNNs_gamma.append(
                GNN(
                    in_dim=in_dim,
                    hidden_dim=2 * Ns_k**2 + 4,
                    out_dim=2 * Ns_k**2,
                    num_layers=3,
                    dp_rate=0.1,
                )
            )
            GNNs_u.append(
                GNN(
                    in_dim=in_dim,
                    hidden_dim=Nt + 10,
                    out_dim=Nt,
                    num_layers=3,
                    dp_rate=0.1,
                )
            )

        ckpt = load_artifact(
            checkpoint_path(I_max),
            map_location="cpu"
        )

        for i in range(I_max):
            GNNs_gamma[i].load_state_dict(ckpt["GNNs_gamma_state_dict"][i])
            GNNs_u[i].load_state_dict(ckpt["GNNs_u_state_dict"][i])

        for m in list(GNNs_gamma) + list(GNNs_u):
            m.eval()

        print(f"[Info] Loaded UWMMSE model (I_max={I_max})")
        return GNNs_gamma, GNNs_u


    UWMMSE_MODELS = None


    # =====================================================
    # Algorithm evaluation routines
    # =====================================================

    def run_classical_WMMSE(prob_inst):
        H_list = prob_inst["H"]
        K = prob_inst["K"]
        Ns = Ns_k * K

        u0 = [torch.ones(Nc, 1, dtype=cdtype)] * Nt
        v0 = [torch.ones(Ns, 1, dtype=cdtype)] * Nt
        for n in range(Nt):
            if torch.linalg.norm(v0[n]) > torch.sqrt(P):
                v0[n] = v0[n] / torch.linalg.norm(v0[n]) * torch.sqrt(P)

        solver = TriHybridWMMSE(
            H_list=H_list,
            Ns_k=[Ns_k] * K,
            Mk=[Mk] * K,
            sigma2=[sigma2] * K,
            beta=[1.0] * K,
            P=[P] * Nt,
            eta=eta,
            u_init=u0,
            v_init=v0,
        )

        _, FA_opt, FD_opt, _, _ = solver.run_classical(
            I_max=200, verbose=False, tol=1e-6
        )

        FRF, FBB = HybridPrecodingBFGS(FD_opt, Nrf).solve(
            max_iter=100, use_B0=True, verbose=False
        )

        _, papc_ok, info = checkHybridConstraints(
            FRF, FBB, P, tol=1e-6, verbose=False
        )
        if not papc_ok:
            FBB = FBB / torch.sqrt(torch.tensor(info["max_papc_ratio"]))

        R, _ = computeWSR(FA_opt, FRF, FBB, H_list, parameters)
        return R.item()


    def run_UWMMSE(dataset, inst_idx, I_max):
        prob_inst = dataset[inst_idx]
        H_list = prob_inst["H_list"]
        K = prob_inst["K"]
        Ns = Ns_k * K

        u0 = [torch.ones(Nc, 1, dtype=cdtype)] * Nt
        v0 = [torch.ones(Ns, 1, dtype=cdtype)] * Nt
        for n in range(Nt):
            if torch.linalg.norm(v0[n]) > torch.sqrt(P):
                v0[n] = v0[n] / torch.linalg.norm(v0[n]) * torch.sqrt(P)

        node_features = prob_inst["node_feature"]
        edge_index = prob_inst["edge_index"]
        graph_id = torch.zeros(K, dtype=torch.int32)
        num_graphs = torch.tensor(1, dtype=torch.int32)

        GNNs_gamma, GNNs_u = UWMMSE_MODELS[I_max]

        delta_gamma_list, delta_u_list = [], []
        for i in range(I_max):
            delta_gamma_list.append(GNNs_gamma[i](node_features, edge_index))
            delta_u = GNNs_u[i](node_features, edge_index, graph_id, num_graphs)
            delta_u_list.append(delta_u.reshape(Nt, 1))

        solver = TriHybridWMMSE(
            H_list=H_list,
            Ns_k=[Ns_k] * K,
            Mk=[Mk] * K,
            sigma2=[sigma2] * K,
            beta=[1.0] * K,
            P=[P] * Nt,
            eta=eta,
            u_init=u0,
            v_init=v0,
        )

        _, FA_opt, FD_opt, _, _ = solver.run_unfolded(
            dgamma_list=delta_gamma_list,
            du_list=delta_u_list,
            I_max=I_max,
            verbose=False
        )

        FRF, FBB = HybridPrecodingBFGS(FD_opt, Nrf).solve(
            max_iter=100, use_B0=True, verbose=False
        )

        _, papc_ok, info = checkHybridConstraints(
            FRF, FBB, P, tol=1e-6, verbose=False
        )
        if not papc_ok:
            FBB = FBB / torch.sqrt(torch.tensor(info["max_papc_ratio"]))

        R, _ = computeWSR(FA_opt, FRF, FBB, H_list, parameters)
        return R.item()


    def run_AOMO_wrapper(prob_inst):
        H_list = prob_inst["H"]
        K = prob_inst["K"]

        H_list_np = [x.numpy() for x in H_list]
        eta_np = [x.numpy().squeeze() for x in eta]

        solver = AOMOBeamformer(
            H=H_list_np,
            Ns_k=[Ns_k]*K,
            sigma2=[sigma2.numpy()]*K,
            beta=[1.0]*K,
            eta=eta_np,
            P=[P.numpy()]*Nt,
            max_iter=30
        )

        FA_opt, FD_opt = solver.solve(verbose=False)
        FA_opt = torch.from_numpy(FA_opt).to(cdtype)
        FD_opt = torch.from_numpy(FD_opt).to(cdtype)

        FRF, FBB = HybridPrecodingBFGS(FD_opt, Nrf).solve(
            max_iter=100, use_B0=True, verbose=False
        )

        _, papc_ok, info = checkHybridConstraints(
            FRF, FBB, P, tol=1e-6, verbose=False
        )
        if not papc_ok:
            FBB = FBB / torch.sqrt(torch.tensor(info["max_papc_ratio"]))

        R, _ = computeWSR(FA_opt, FRF, FBB, H_list, parameters)
        return R.item()


    def run_Decomp_wrapper(prob_inst):
        H_list = prob_inst["H"]
        K = prob_inst["K"]

        H_list_np = [x.numpy() for x in H_list]
        eta_np = [x.numpy().squeeze() for x in eta]

        solver = DecompBeamformer(
            H=H_list_np,
            Ns_k=[Ns_k]*K,
            sigma2=[sigma2.numpy()]*K,
            beta=[1.0]*K,
            eta=eta_np,
            P=[P.numpy()]*Nt,
            max_iterP1=30,
            max_iterP2=30
        )

        FA_opt, FD_opt = solver.solve(verbose=False)
        FA_opt = torch.from_numpy(FA_opt).to(cdtype)
        FD_opt = torch.from_numpy(FD_opt).to(cdtype)

        FRF, FBB = HybridPrecodingBFGS(FD_opt, Nrf).solve(
            max_iter=100, use_B0=True, verbose=False
        )

        _, papc_ok, info = checkHybridConstraints(
            FRF, FBB, P, tol=1e-6, verbose=False
        )
        if not papc_ok:
            FBB = FBB / torch.sqrt(torch.tensor(info["max_papc_ratio"]))

        R, _ = computeWSR(FA_opt, FRF, FBB, H_list, parameters)
        return R.item()


    # =====================================================
    # Algorithms list
    # =====================================================
    ALG_NAMES = ( [f"Unfolded-{I}" for I in UWMMSE_I_LIST] + ["Model-Based", "AOMO", "Decomp"] )


    # =====================================================
    # Main loop
    # =====================================================
    mean_results = {s: {a: 0.0 for a in ALG_NAMES} for s in SCENARIOS}

    for scenario, fname in SCENARIOS.items():

        print("\n==============================")
        print("Scenario:", scenario)
        print("==============================")

        data = load_artifact(os.path.join(DATA_DIR, fname))
        ProbInsts = data["ProbInsts"][:cfg["max_instances"]]

        dataset = channelDataset(ProbInsts, parameters)

        if UWMMSE_MODELS is None:
            example_H = ProbInsts[0]["H"][0]
            UWMMSE_MODELS = {
                I: load_uwmmse_models(I, example_H)
                for I in UWMMSE_I_LIST
            }

        scores = {a: [] for a in ALG_NAMES}

        for idx in tqdm(range(len(ProbInsts))):

            scores["Model-Based"].append(
                run_classical_WMMSE(ProbInsts[idx])
            )

            for I in UWMMSE_I_LIST:
                scores[f"Unfolded-{I}"].append(
                    run_UWMMSE(dataset, idx, I))
            
            scores["AOMO"].append(run_AOMO_wrapper(ProbInsts[idx]))
            scores["Decomp"].append(run_Decomp_wrapper(ProbInsts[idx]))

        for a in ALG_NAMES:
            mean_results[scenario][a] = np.mean(scores[a])


    # =====================================================
    # Save results
    # =====================================================
    np.save(
        os.path.join(RESULT_DIR, "mean_WSR_results.npy"),
        mean_results
    )

    print("\nResults saved to:", RESULT_DIR)


In [ ]:
# =====================================================
# PART 2 — LOAD RESULTS AND PLOT
# =====================================================

import os
import numpy as np
import matplotlib.pyplot as plt


RESULT_DIR = result_dir("results_generalization_barplot")
RESULT_FILE = os.path.join(RESULT_DIR, "mean_WSR_results.npy")

mean_results = np.load(RESULT_FILE, allow_pickle=True).item()

scenarios = list(mean_results.keys())
ALG_NAMES = list(next(iter(mean_results.values())).keys())

num_alg = len(ALG_NAMES)

x = np.arange(len(scenarios))
width = 0.12

plt.figure(figsize=(10,6))

for i, alg in enumerate(ALG_NAMES):
    vals = [mean_results[s][alg] for s in scenarios]
    plt.bar(
        x + (i - num_alg/2)*width + width/2,
        vals,
        width,
        label=alg
    )

plt.xticks(x, scenarios)
plt.ylabel("Mean Optimized WSR (bit/s/Hz)")
plt.title("Algorithm Generalization Across Scenarios")
plt.legend(ncol=2)
plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(
    os.path.join(RESULT_DIR, "generalization_barplot.png"),
    dpi=300
)
plt.show()

print("Figure saved to:", RESULT_DIR)


In [ ]:
# =====================================================
# EXPORT GROUPED BAR PLOT TO TIKZ
# =====================================================

import os
import numpy as np

RESULT_DIR = result_dir("results_generalization_barplot")
RESULT_FILE = os.path.join(RESULT_DIR, "mean_WSR_results.npy")

mean_results = np.load(RESULT_FILE, allow_pickle=True).item()

scenarios = list(mean_results.keys())
ALG_NAMES = list(next(iter(mean_results.values())).keys())


# =====================================================
# Print TikZ axis header (optional but recommended)
# =====================================================
print("% ==================================================")
print("% TIKZ GROUPED BAR PLOT")
print("% paste inside tikzpicture")
print("% ==================================================")
print()

print("\\begin{axis}[")
print("    ybar,")
print("    bar width=6pt,")
print("    width=3.4in,")
print("    height=2.2in,")
print("    ymin=0,")
print("    ylabel={Mean Optimized WSR (bit/s/Hz)},")
print("    symbolic x coords={", ", ".join(scenarios), "},")
print("    xtick=data,")
print("    x tick label style={rotate=0, anchor=center},")
print("    legend style={at={(0.5,1.05)}, anchor=south, legend columns=2},")
print("    ymajorgrids,")
print("    grid style=dashed,")
print("]")
print()


# =====================================================
# Print bars
# =====================================================
for alg in ALG_NAMES:

    print(f"% ---- {alg} ----")
    print("\\addplot coordinates {")

    for s in scenarios:
        val = mean_results[s][alg]
        print(f"({s}, {val:.6f})")

    print("};")
    print(f"\\addlegendentry{{{alg}}}")
    print()


print("\\end{axis}")
